# Chronos-2: Rolling M+1 backtest (3 months) + 12-month forecast

This notebook defines two functions **assuming there are NO known future covariates** (i.e., we do not pass any `future_covariates` to Chronos-2).

1) `run_mplus1_backtest_3months(...)`: rolling-origin backtest for the last 3 target months using an **M+1 with 1-month data latency** convention (as in your example).

2) `forecast_next_12_months(...)`: trains on all available history (up to the latest month in the file) and forecasts the next 12 months.


In [ ]:
# If needed (Databricks/Jupyter), install dependencies
%pip install chronos-forecasting>=2.2[extras] matplotlib
# If running on Databricks, %pip is recommended. In regular Jupyter, %pip also works (IPython).
%pip install -q "autogluon.timeseries>=1.1.0" "torch>=2.1.0" "transformers>=4.40.0" huggingface_hub mlflow pyyaml pandas numpy pyarrow

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from chronos import BaseChronosPipeline, Chronos2Pipeline

In [ ]:
# Choose device
device_map = "cuda" if (os.environ.get("CUDA_VISIBLE_DEVICES", "") != "" or True) else "cpu"
# If you know you are on CPU-only, set: device_map = "cpu"

PIPELINE: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device_map,
)
print("Loaded Chronos-2 on", device_map)

In [ ]:
def _to_month_start(dt: pd.Timestamp) -> pd.Timestamp:
    dt = pd.Timestamp(dt)
    return dt.to_period("M").to_timestamp(how="start")

def _read_preprocessed(path: str) -> pd.DataFrame:
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    if path.lower().endswith(".parquet"):
        return pd.read_parquet(path)
    raise ValueError("Unsupported file type. Please provide a .csv or .parquet preprocessed file.")

def _to_np(x):
    '''Convert torch / list / nested outputs into numpy array.'''
    if hasattr(x, "detach"):          # torch tensor
        return x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return np.array([_to_np(xx) for xx in x], dtype=object)
    return np.asarray(x)

def _squeeze_to_1d(arr, H: int) -> np.ndarray:
    arr = _to_np(arr)
    arr = np.squeeze(arr)
    if arr.ndim == 0:
        arr = np.repeat(arr, H)
    arr = np.squeeze(arr)
    if arr.ndim != 1 or arr.shape[0] != H:
        raise ValueError(f"Unexpected mean shape: {arr.shape} (expected ({H},))")
    return arr.astype(np.float32)

def _squeeze_to_2d(arr, Q: int, H: int) -> np.ndarray:
    arr = _to_np(arr)
    arr = np.squeeze(arr)

    if arr.ndim == 2:
        if arr.shape == (Q, H):
            out = arr
        elif arr.shape == (H, Q):
            out = arr.T
        else:
            raise ValueError(f"Unexpected quantiles shape: {arr.shape} (expected ({Q},{H}) or ({H},{Q}))")
    elif arr.ndim == 1 and Q == 1 and arr.shape[0] == H:
        out = arr.reshape(1, H)
    elif arr.ndim == 3 and arr.shape[-1] == 1:
        out = np.squeeze(arr[..., 0])
        if out.shape == (H, Q):
            out = out.T
        if out.shape != (Q, H):
            raise ValueError(f"Unexpected quantiles shape after squeeze: {out.shape}")
    else:
        raise ValueError(f"Unexpected quantiles array ndim={arr.ndim}, shape={arr.shape}")

    return out.astype(np.float32)

def get_target_months(latest_month: pd.Timestamp, backtest_months: int) -> list[pd.Timestamp]:
    latest_month = _to_month_start(latest_month)
    if backtest_months < 1:
        raise ValueError("backtest_months must be >= 1")
    # last N months ending at latest_month: [latest-(N-1), ..., latest]
    return [_to_month_start(latest_month - pd.DateOffset(months=m))
            for m in range(backtest_months - 1, -1, -1)]

In [ ]:
def run_mplus1_backtest_3months(
    preprocessed_df: pd.DataFrame,
    *,
    time_col: str = "date",
    item_id_col: str = "sku",
    target_col: str = "Volume",
    backtest_months: int = 3,
    # These are treated as *past* covariates only (no known future covariates are assumed).
    past_covariates: list[str] | None = None,
    static_categorical: list[str] | None = None,
    quantiles: list[float] = [0.1, 0.5, 0.9],
    min_history: int = 12,
    batch_size: int = 128,
    pipeline: Chronos2Pipeline | None = None,
    ) -> pd.DataFrame:
    '''
    Rolling-origin backtest for the last 3 target months using your example logic,
    assuming **no known future covariates**.

    Example (latest month = Dec):
      - Train through Aug (Dec-4) and forecast Oct (Dec-2)  -> 2-step ahead
      - Train through Sep (Dec-3) and forecast Nov (Dec-1)  -> 2-step ahead
      - Train through Oct (Dec-2) and forecast Dec (Dec)    -> 2-step ahead

    Saves ONE CSV with forecasts + actuals for all 3 target months.

    Inputs expected:
      - One row per (item_id_col, time_col) month
      - target_col is the numeric target
      - Covariates are used only if you pass column lists via `past_covariates`
        and/or `static_categorical`. If you leave them as None, we assume **no covariates**.
    '''
    if pipeline is None:
        pipeline = PIPELINE

    df = preprocessed_df

    for col in (time_col, item_id_col, target_col):
        if col not in df.columns:
            raise ValueError(f"'{col}' not found in input columns.")

    # If you don't provide covariate column lists, we assume **no covariates**.
    # Pass explicit lists if you want to use them.
    past_covariates = past_covariates or []
    static_categorical = static_categorical or []

    # Keep only columns that exist in the file
    past_covariates = [c for c in past_covariates if c in df.columns]
    static_categorical = [c for c in static_categorical if c in df.columns]

    df[time_col] = pd.to_datetime(df[time_col]).map(_to_month_start)
    df = df.sort_values([item_id_col, time_col])

    latest_month = _to_month_start(df[time_col].max())
    target_months = get_target_months(latest_month, backtest_months=backtest_months)

    pred_len = 2  # because train_end = target_month - 2 months (1-month latency), so we need 2-step-ahead
    Q = len(quantiles)

    tasks: list[dict] = []

    for target_month in target_months:
        train_end = _to_month_start(target_month - pd.DateOffset(months=2))
        print("\n Creating records for Target month:", target_month.strftime("%Y-%m"))

        for key, g in df.groupby(item_id_col, sort=False):
            g = g.sort_values(time_col)

            hist = g[g[time_col] <= train_end]
            if len(hist) < min_history:
                continue

            act_row = g[g[time_col] == target_month]
            if act_row.empty:
                continue
            actual_target = float(act_row[target_col].iloc[0])

            target = hist[target_col].astype(np.float32).to_numpy()
            input_obj: dict = {"target": target}

            # Past covariates only (no future covariates passed)
            past_covs: dict = {}
            for c in past_covariates:
                past_covs[c] = hist[c].astype(np.float32).to_numpy()
            for c in static_categorical:
                past_covs[c] = hist[c].astype(str).to_numpy()

            if past_covs:
                input_obj["past_covariates"] = past_covs

            tasks.append({
                "key": key,
                "target_month": target_month,
                "train_end": train_end,
                "actual": actual_target,
                "input": input_obj,
            })

    if not tasks:
        raise ValueError("No valid series found for backtest. Check date coverage, min_history, and required columns.")

    rows: list[dict] = []

    def chunker(seq, size):
        for i in range(0, len(seq), size):
            yield seq[i:i+size]

    

    for batch in tqdm(chunker(tasks, batch_size), total=(len(tasks) + batch_size - 1) // batch_size):
        inputs = [t["input"] for t in batch]
        quantiles_list, mean_list = pipeline.predict_quantiles(
            inputs,
            prediction_length=pred_len,
            quantile_levels=quantiles,
        )

        for i, t in enumerate(batch):
            q_arr = _squeeze_to_2d(quantiles_list[i], Q, pred_len)
            m_arr = _squeeze_to_1d(mean_list[i], pred_len)

            step_idx = pred_len - 1  # the target month (2nd step)
            out = {
                item_id_col: t["key"],
                "train_end": t["train_end"],
                "target_month": t["target_month"],
                "actual": t["actual"],
                "mean": float(m_arr[step_idx]),
            }
            for qi, q in enumerate(quantiles):
                out[f"q{int(round(q*100)):02d}"] = float(q_arr[qi, step_idx])

            rows.append(out)

    out_df = pd.DataFrame(rows).sort_values([item_id_col, "target_month"])
    # out_df.to_csv(output_csv_path, index=False)
    print("\n Backtest completed")
    return out_df


In [ ]:
def forecast_next_12_months(
    preprocessed_df: pd.DataFrame,
    # output_csv_path: str,
    *,
    time_col: str = "date",
    item_id_col: str = "sku",
    target_col: str = "Volume",
    # These are treated as *past* covariates only (no known future covariates are assumed).
    past_covariates: list[str] | None = None,
    static_categorical: list[str] | None = None,
    quantiles: list[float] = [0.1, 0.5, 0.9],
    min_history: int = 12,
    batch_size: int = 128,
    pipeline: Chronos2Pipeline | None = None,
    prediction_length: int = 12,
    ) -> pd.DataFrame:
    '''
    Train on all available history (up to the latest month in the file) and forecast
    the next `prediction_length` months, assuming **no known future covariates**.

    Notes:
      - Past covariates / static categoricals are used only if you pass column lists.
      - No future covariates are passed to Chronos-2 (so you do NOT need to generate
        future values for any covariate columns).
    '''
    print("\n Forecasting next 12 months")
    if pipeline is None:
        pipeline = PIPELINE

    df = preprocessed_df

    for col in (time_col, item_id_col, target_col):
        if col not in df.columns:
            raise ValueError(f"'{col}' not found in input columns.")

    # If you don't provide covariate column lists, we assume **no covariates**.
    # Pass explicit lists if you want to use them.
    past_covariates = past_covariates or []
    static_categorical = static_categorical or []

    # Keep only columns that exist in the file
    past_covariates = [c for c in past_covariates if c in df.columns]
    static_categorical = [c for c in static_categorical if c in df.columns]

    df[time_col] = pd.to_datetime(df[time_col]).map(_to_month_start)
    df = df.sort_values([item_id_col, time_col])

    latest_month = _to_month_start(df[time_col].max())
    future_dates = pd.date_range(latest_month + pd.DateOffset(months=1), periods=prediction_length, freq="MS")
    Q = len(quantiles)

    tasks: list[dict] = []

    for key, g in df.groupby(item_id_col, sort=False):
        g = g.sort_values(time_col)
        if len(g) < min_history:
            continue

        target = g[target_col].astype(np.float32).to_numpy()
        input_obj: dict = {"target": target}

        # Past covariates only (no future covariates passed)
        past_covs: dict = {}
        for c in past_covariates:
            past_covs[c] = g[c].astype(np.float32).to_numpy()
        for c in static_categorical:
            past_covs[c] = g[c].astype(str).to_numpy()

        if past_covs:
            input_obj["past_covariates"] = past_covs

        tasks.append({"key": key, "input": input_obj})

    if not tasks:
        raise ValueError("No valid series found for forecasting. Check min_history and required columns.")

    groups = list(df.groupby(ITEM_ID_COL, sort=False))
    print("\nKeys:", len(groups)) 
    print("\nPrepared inputs:", len(tasks))

    rows: list[dict] = []

    def chunker(seq, size):
        for i in range(0, len(seq), size):
            yield seq[i:i+size]

    for batch in chunker(tasks, batch_size):
        inputs = [t["input"] for t in batch]
        quantiles_list, mean_list = pipeline.predict_quantiles(
            inputs,
            prediction_length=prediction_length,
            quantile_levels=quantiles,
        )

        for i, t in enumerate(batch):
            q_arr = _squeeze_to_2d(quantiles_list[i], Q, prediction_length)
            m_arr = _squeeze_to_1d(mean_list[i], prediction_length)

            for h in range(prediction_length):
                out = {
                    item_id_col: t["key"],
                    "train_end": latest_month,
                    "forecast_month": future_dates[h],
                    "mean": float(m_arr[h]),
                }
                for qi, q in enumerate(quantiles):
                    out[f"q{int(round(q*100)):02d}"] = float(q_arr[qi, h])
                rows.append(out)

    out_df = pd.DataFrame(rows).sort_values([item_id_col, "forecast_month"])
    # out_df.to_csv(output_csv_path, index=False)
    return out_df


## Actual usage

Update the below to run backtest and forecasts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# raw = pd.read_csv(os.environ["BLOB_URL"])  # old dataset

raw = pd.read_csv(os.environ["BLOB_URL"])

raw.head()

In [ ]:
raw.columns.to_list()

In [ ]:

# --- Perform all renaming and initial manipulation ---

# Converting dummies to categoricals 
df=raw
df = df.rename(columns={"Category": "cat_bucket"})

qtr_buckets = ['Quarter_1','Quarter_2','Quarter_3','Quarter_4',]
df['qtr_bucket'] = df[qtr_buckets].idxmax(axis=1).str.replace('Quarter_', '')
df = df.drop(columns=qtr_buckets)

# Selecting and renaming columns
df = df[['Date',
         'key',
  'Sales_Volume',
  'Sales_Value',
  'BTL_Discount',
  'btl_lag1',
  'btl_lag2',
  'btl_lag3',
  'BTL_Discount',
  'Per_BTL_change',
  'vol_zero_count_L2to4m',
  'vol_zero_count_L2to7m',
  'vol_zero_count_L2to13m',
  'months_since_1st_existance',
  'temp',
  'humidity',
  'precip',
  'ASM_saliency_month',
  'ASM_saliency_qtr',
  'ASM_saliency_half_year',
  'DT_saliency_month',
  'DT_saliency_qtr',
  'DT_saliency_half_year',
  'Category_saliency_month',
  'Category_saliency_qtr',
  'Category_saliency_half_year',
  'AOP_saliency_month',
  'AOP_saliency_qtr',
  'AOP_saliency_half_year',
  'qtr_bucket',
  'ASM',
  'DT',
  'AOP_Track',
  'cat_bucket',
  'State',
  'SubCategory',
  'Mother Brand',
  'GRP',
  'SOV',
  'L3M_moving_avg_forecast',
  'L4M_moving_avg_forecast',
  'SMLY_sales_forecast',
  'GRP_lag1',
  'GRP_lag2',
  'SOV_lag1',
  'SOV_lag2',
  'SOV_lag14',
  'SOV_lag17',
  'GRP_SOV_Interaction',
  'GRP_SOV_Interaction_lag1',
  'GDP_in_lkh_cr',
  'per_change_gdp',
  'per_capita_income',
  'per_change_income',
  'Inflation',
  'Inflation_Lag3',
  'INR_to_USD_fluctuation',
  'Population_in_000',
  'Avg_Inflation_FY',
  'Income_Inflation_Index',
  'MRP',
  'MRP_lag1',
  'MRP_lag2'
]]

df.columns = ['Date',
              'key',
  'Sales vol - Cases',
  'Sales_Value',
  'BTL_Discount',
  'btl_lag1',
  'btl_lag2',
  'btl_lag3',
  'BTL%',
  'Per_BTL_change',
  'vol_zero_count_L2to4m',
  'vol_zero_count_L2to7m',
  'vol_zero_count_L2to13m',
  'months_since_1st_existance',
  'Temp',
  'humidity',
  'precipitation',
  'ASM_saliency_month',
  'ASM_saliency_qtr',
  'ASM_saliency_half_year',
  'DT_saliency_month',
  'DT_saliency_qtr',
  'DT_saliency_half_year',
  'Category_saliency_month',
  'Category_saliency_qtr',
  'Category_saliency_half_year',
  'AOP_saliency_month',
  'AOP_saliency_qtr',
  'AOP_saliency_half_year',
  'qtr_bucket',
  'State',
  'SKU',
  'SKU format',
  'SKU category',
  'State group',
  'SKU subcategory',
  'SKU brand',
  'GRP',
  'SOV',
  'L3M_moving_avg_forecast',
  'L4M_moving_avg_forecast',
  'SMLY_sales_forecast',
  'GRP_lag1',
  'GRP_lag2',
  'SOV_lag1',
  'SOV_lag2',
  'SOV_lag14',
  'SOV_lag17',
  'GRP_SOV_Interaction',
  'GRP_SOV_Interaction_lag1',
  'GDP',
  'per_change_gdp',
  'per_capita_income',
  'per_change_income',
  'Inflation',
  'Inflation_Lag3',
  'INR_to_USD_fluctuation',
  'Population_in_000',
  'Avg_Inflation_FY',
  'Income_Inflation_Index',
  'Price',
  'MRP_lag1',
  'MRP_lag2',
]

# print("\nConversion of dummies to categoricals completed")

# # Mapping Geo heirarchy
# reg = spark.sql("""select SalesOfficeCode, 
#        first(SalesDistrictDescription) as Region
# from prod.curated_plus.dim_customer
# where SalesDistrictDescription is not null
#   and SalesDistrictDescription not like '%OTC%'
# group by SalesOfficeCode""")
# reg_pd = reg.toPandas()
# df = df.merge(reg_pd, how="left", left_on="State", right_on="SalesOfficeCode")
# df = df.rename(columns={"State_x": "State"})

# print("\nMapping Geo heirarchy completed")

# Mapping product heirarchy
mat = spark.sql("""select dm.DesignType,
  first(dmh.BrandDescription) as Brand,
  first(dmh.SubCategoryDescription) as  SubCategory
  from prod.curated_plus.dim_material dm
  left join prod.curated.dim_materialhierarchy dmh
  on dm.MaterialHierarchy = dmh.MaterialHierarchy
  where dmh.IsActive = 1
  and dmh.SubCategoryDescription is not null
  and dmh.BrandDescription is not null
  group by dm.DesignType""")
mat_pd = mat.toPandas()
df = df.merge(mat_pd, how="left", left_on="SKU", right_on="DesignType")
df = df.drop(columns=["SKU subcategory"])
df = df.rename(columns={"SubCategory": "SKU subcategory"})
df = df.drop(columns=["SKU brand"])
df = df.rename(columns={"Brand": "SKU brand"})

print("\nMapping product heirarchy completed")

df.head()

In [ ]:
# Cell 3: config (use your lists exactly)
# INPUT_TABLE = "main.default.your_table"  # <-- change this to your Spark table

TIME_COL = "Date"
ITEM_ID_COL = "key"
TARGET_COL = "Sales vol - Cases"
H = 4
FREQ = "MS"   # monthly month-start

KNOWN_COVARIATES = [
    'btl_lag1','btl_lag2','btl_lag3','BTL%','Per_BTL_change',
    'vol_zero_count_L2to4m','vol_zero_count_L2to7m','vol_zero_count_L2to13m',
    'months_since_1st_existance',
    'Temp','humidity','precipitation',
    'ASM_saliency_month','ASM_saliency_qtr','ASM_saliency_half_year',
    'DT_saliency_month','DT_saliency_qtr','DT_saliency_half_year',
    'Category_saliency_month','Category_saliency_qtr','Category_saliency_half_year',
    'AOP_saliency_month','AOP_saliency_qtr','AOP_saliency_half_year',
    'GRP','SOV','L3M_moving_avg_forecast','L4M_moving_avg_forecast','SMLY_sales_forecast',
    'GRP_lag1','GRP_lag2','SOV_lag1','SOV_lag2','SOV_lag14','SOV_lag17',
    'GRP_SOV_Interaction','GRP_SOV_Interaction_lag1',
    'GDP','per_change_gdp','per_capita_income','per_change_income',
    'Inflation','Inflation_Lag3','INR_to_USD_fluctuation','Population_in_000',
    'Avg_Inflation_FY','Income_Inflation_Index',
    'Price','MRP_lag1','MRP_lag2'
]

"Date","SKU", "SKU format", "SKU brand", "SKU subcategory","SKU category", "State", "State group","Sales vol - Cases",'BTL%','Temp','humidity','precipitation','GRP','SOV','Price'

STATIC_CATEGORICAL = [
    "SKU", "SKU format", "SKU brand", "SKU subcategory",
    "SKU category", "State", "State group"
]

MIN_HISTORY = 12  # drop series with < 12 months history (adjust)
BATCH_SIZE = 128  # inference batching for 10k series
QUANTILES = [0.1, 0.5, 0.9]

df = df[df[TIME_COL] <= '2026-01-01']


In [ ]:
# Example:
# preprocessed_path = os.path.join(os.environ["DATABRICKS_OUTPUT_DIR"], "RSA_ADS_2026-01-01.csv")
# backtest_out = os.path.join(os.environ["DATABRICKS_OUTPUT_DIR"], "rsa_backtest_dec25.csv")
# forecast_out  = os.path.join(os.environ["DATABRICKS_OUTPUT_DIR"], "rsa_pred_jan26_dec26.csv")
#
bt_df = run_mplus1_backtest_3months(df, time_col=TIME_COL,item_id_col=ITEM_ID_COL,target_col=TARGET_COL, past_covariates=KNOWN_COVARIATES,static_categorical=STATIC_CATEGORICAL,backtest_months=6)
fc_df = forecast_next_12_months(df,time_col=TIME_COL,item_id_col=ITEM_ID_COL,target_col=TARGET_COL,past_covariates=KNOWN_COVARIATES,static_categorical=STATIC_CATEGORICAL, prediction_length=13)

display(bt_df.head())
display(fc_df.head())

In [ ]:
output_dir = os.environ["DATABRICKS_OUTPUT_DIR"]
bt_df.to_csv(f"{output_dir}/india_backtest_oldADS_6mnth_Jan26.csv", index=False)
fc_df.to_csv(f"{output_dir}/India_FC_Jan26_Jan27.csv", index=False)


In [ ]:
display(df[['SKU','SKU format','SKU category']].drop_duplicates())

In [ ]:
output_dir = os.environ["DATABRICKS_OUTPUT_DIR"]
df[["Date","SKU", "SKU format", "SKU brand", "SKU subcategory","SKU category", "State", "State group","Sales vol - Cases",'Sales_Value',
  'BTL_Discount','Temp','humidity','precipitation']].to_csv(f"{output_dir}/India_ADS_Jan26.csv")

In [ ]:
%sql
select SalesOfficeCode, 
        first(SalesDistrictDescription) as Region
 from prod.curated_plus.dim_customer
 where SalesDistrictDescription is not null
   and SalesDistrictDescription not like '%OTC%' group by SalesOfficeCode